# Tome 6, Chapter 5: Baire's Theorem (Python companion)

Exercises first, solutions at the end. Typical objects in two senses: incompressible strings and Kolmogorov complexity, the slopes of Takagi's function as a random walk, the roughness of Weierstrass's function, continued fractions of Liouville numbers against Khinchin's law, Thomae's function, and a typical number built by the proof of Baire's theorem. Requirements: numpy, mpmath and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P5.1 (★★) CS application: most strings are incompressible.** (a) Show by counting that fewer than a fraction $2^{-k}$ of the $n$-bit strings can be described by a program shorter than $n-k$ bits, whatever the programming language. (b) Compress with `zlib`, `bz2` and `lzma` a million random bits, the binary digits of $\sqrt2$, a million bits of Champernowne's sequence $1\,10\,11\,100\ldots$, and the text of this exercise repeated. Which compress? (c) Compare with Proposition 5.14: what does a *typical* sequence in the sense of Baire look like to a compressor?

**P5.2 (★) Takagi's slopes are a random walk.** For $x\in[0,1[$, compute the slopes $s_k$ of the proof of Theorem 5.6 and show that $s_k=\sum_{n<k}(1-2d_{n+1})$ where $d_1d_2\ldots$ are the binary digits of $x$. Plot $s_k$ for a few random $x$ and for $x=\frac13$.

**P5.3 (★) How rough is Weierstrass's function?** For $W(x)=\sum_na^n\cos(b^n\pi x)$ with $a=\frac12$, $b=3$, estimate $\omega(h)=\max_{|x-y|\le h}|W(x)-W(y)|$ for $h=3^{-1},\ldots,3^{-7}$ and fit $\omega(h)\approx Ch^\alpha$. Compare $\alpha$ with $\frac{\log(1/a)}{\log b}$.

**P5.4 (★) Liouville numbers and continued fractions.** Compute the continued fraction expansions of Liouville's constant $\sum_k10^{-k!}$, of $\pi$, of $e$ and of a random number (with $200$-digit arithmetic). What distinguishes a Liouville number? For random numbers, test Khinchin's law: the geometric mean of the partial quotients tends to $2.685\ldots$ for almost every $x$.

**P5.5 (★) Thomae's function.** Plot $f(\frac pq)=\frac1q$ on $[0,1]$ for $q\le 60$. Then check numerically the continuity at $\sqrt2-1$: compute $\max\{f(y):|y-(\sqrt2-1)|<\delta\}$ for $\delta=10^{-1},\dots,10^{-6}$. How does it relate to the continued fraction of $\sqrt2$?

**P5.6 (★★) A typical number in Baire's sense.** Build a point of the comeagre set of Proposition 5.14 by the procedure of the proof of Baire's theorem: alternately append a few random digits (to enter an arbitrary open set) and a long run of ones (to enter $V_N$). Plot $\nu_n(x)$ against $n$ on a log scale, and compare with a random (normal) number.

## Solutions

### P5.1

In [ ]:
import zlib, bz2, lzma, numpy as np
from mpmath import mp, sqrt

n_bits = 10 ** 6
rng = np.random.default_rng(0)
random_bits = rng.integers(0, 2, n_bits, dtype=np.uint8)

mp.prec = n_bits + 10
s2 = int(sqrt(2) * 2 ** n_bits)                      # floor(sqrt(2) 2^N)
sqrt2_bits = np.array(list(bin(s2)[3:3 + n_bits]), dtype=np.uint8)

champ = []
k = 1
while len(champ) < n_bits:
    champ.extend(int(c) for c in bin(k)[2:]); k += 1
champ = np.array(champ[:n_bits], dtype=np.uint8)

text = ("Compress with zlib, bz2 and lzma a million random bits. " * 3000).encode()
def pack(bits): return np.packbits(bits).tobytes()
for name, data in [("random", pack(random_bits)), ("sqrt 2", pack(sqrt2_bits)), ("Champernowne", pack(champ)), ("repeated text", text)]:
    sizes = [len(zlib.compress(data, 9)), len(bz2.compress(data, 9)), len(lzma.compress(data))]
    print(f"{name:14s} {len(data):7d} bytes -> zlib {sizes[0]/len(data):.3f}  bz2 {sizes[1]/len(data):.3f}  lzma {sizes[2]/len(data):.3f}")

(a) There are $2^0+2^1+\cdots+2^{n-k-1}<2^{n-k}$ binary programs of length less than $n-k$, and each describes at most one string; so at most a fraction $2^{-k}$ of the $2^n$ strings of length $n$ have a description shorter than $n-k$. Most strings are *incompressible*: their Kolmogorov complexity is close to $n$. (b) The random bits do not compress at all (ratios slightly above $1$). Neither do the digits of $\sqrt2$, although a ten-line program generates them: their Kolmogorov complexity is tiny, but no general-purpose compressor finds the program. Kolmogorov complexity is uncomputable, by a diagonal argument like the halting problem of Chapter 1. Champernowne's sequence compresses a little (its blocks have visible structure), and the repeated text compresses enormously. (c) Incompressibility is a *measure*-typical property: random sequences have it with probability one. In the sense of category the typical sequence is the opposite: it contains, infinitely often, enormous runs of ones (P5.6), so infinitely many of its prefixes compress almost to nothing. Category favours structure; measure favours chaos.

### P5.2

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from fractions import Fraction

phi = lambda y: min(y - np.floor(y), np.ceil(y) - y)
def takagi(x, N=60): return sum(float(phi(Fraction(2) ** n * x)) / 2 ** n for n in range(N))

def slopes(x, K):
    out = []
    for k in range(1, K + 1):
        j = int(x * 2 ** k)
        u, v = Fraction(j, 2 ** k), Fraction(j + 1, 2 ** k)
        out.append(round((takagi(v) - takagi(u)) * 2 ** k))
    return out

def digits(x, K):
    d = []
    for _ in range(K):
        x *= 2; b = int(x); d.append(b); x -= b
    return d

x = Fraction(np.random.default_rng(3).integers(0, 2 ** 40), 2 ** 40)
d = digits(x, 30)
print("slopes from T:     ", slopes(x, 30))
print("from binary digits:", [int(v) for v in np.cumsum([1 - 2 * b for b in d])])

rng = np.random.default_rng(1)
for _ in range(4):
    d = rng.integers(0, 2, 400)
    plt.plot(np.cumsum(1 - 2 * d), lw=0.8)
plt.plot(np.cumsum([1 - 2 * b for b in digits(Fraction(1, 3), 400)]), "k", lw=1.5, label="x = 1/3")
plt.legend(); plt.xlabel("k"); plt.ylabel("s_k"); plt.show()

The $n$-th term $2^{-n}\phi(2^nx)$ rises with slope $+1$ when the fractional part of $2^nx$ is below $\frac12$, that is, when the binary digit $d_{n+1}$ is $0$, and falls with slope $-1$ when $d_{n+1}=1$. So $s_k$ is the number of zeros minus the number of ones among $d_1,\dots,d_k$: for a uniform $x$, a simple random walk (Tome 3). It never converges; it does not even stay bounded, and by the law of the iterated logarithm it reaches $\pm\sqrt{2k\log\log k}$ infinitely often. For $x=\frac13=0.010101\ldots_2$ the walk oscillates between $0$ and $1$: the difference quotients alternate and $T$ has no derivative there either, just as the proof says for every $x$.

### P5.3

In [ ]:
import numpy as np
a, b, N = 0.5, 3, 16
x = np.linspace(0, 2, 3 ** 10 * 2 + 1)
W = sum(a ** n * np.cos(b ** n * np.pi * x) for n in range(N))
hs, oms = [], []
for j in range(1, 9):
    step = 3 ** (10 - j)                      # h = 3^-j in grid units
    om = max(np.max(np.abs(W[s:] - W[:-s])) for s in range(1, step + 1, max(1, step // 20)))
    hs.append(3.0 ** -j); oms.append(om)
    print(f"h = 3^-{j}: omega = {om:.4f}")
print("ratios omega(h)/omega(h/3):", [round(float(oms[i] / oms[i + 1]), 3) for i in range(len(oms) - 1)])
alpha = np.polyfit(np.log(hs[3:]), np.log(oms[3:]), 1)[0]
print("fitted alpha on the small scales =", round(alpha, 3), "  log(1/a)/log(b) =", round(np.log(2) / np.log(3), 3))

The oscillation over windows of length $h$ scales like $h^\alpha$: each division of $h$ by $3$ divides $\omega(h)$ by a ratio close to $2=3^{0.631}$, and the fitted exponent on the small scales is near $\frac{\log2}{\log3}\approx0.63$ (a little lower, since the sampled maximum underestimates $\omega$ and only $16$ terms are kept). $W$ is Hölder of exponent $\frac{\log2}{\log3}$ and of no larger exponent. Heuristically, the terms with $b^n\approx\frac1h$ contribute $a^n=h^{\log(1/a)/\log b}$, the earlier ones are smooth on that scale and the later ones are smaller. A differentiable function would have $\alpha=1$. The graph has box-counting dimension $2-\alpha\approx1.37$ (Chapter 11); that its Hausdorff dimension is also $2-\alpha$ was proved only in 2018 by Shen.

### P5.4

In [ ]:
from mpmath import mp, mpf, pi, e, floor, factorial, log, exp, rand
mp.dps = 400

def cf(x, n):
    out = []
    for _ in range(n):
        a = int(floor(x)); out.append(a); x = x - a
        if x == 0: break
        x = 1 / x
    return out

L = sum(mpf(10) ** (-int(factorial(k))) for k in range(1, 6))   # the next term, 10^-720, is beyond this precision
print("Liouville:", [q if q < 10 ** 6 else f"<{len(str(q))} digits>" for q in cf(L, 22)])
print("pi:       ", cf(pi, 20))
print("e:        ", cf(e, 20))
for trial in range(3):
    q = cf(rand(), 300)[1:]
    gm = exp(sum(log(t) for t in q) / len(q))
    print(f"random #{trial}: geometric mean of 300 partial quotients = {float(gm):.3f}")

Liouville's constant has partial quotients that explode: among small ones, a quotient with $12$ digits appears, then one with $72$ digits (the next would have $480$): one huge quotient for each gap between consecutive terms $10^{-k!}$ and $10^{-(k+1)!}$, with about $(k+1)!-2k!$ digits. Huge partial quotients are exactly what makes a number extremely well approximable: if $a_{k+1}$ is the next quotient, the convergent $\frac{p_k}{q_k}$ satisfies $|x-\frac{p_k}{q_k}|<\frac1{a_{k+1}q_k^2}$ (Tome 1, Chapter 8). The partial quotients of $\pi$ look random (with the famous $292$), those of $e$ follow the pattern $1,2k,1$. For random numbers the geometric mean is close to Khinchin's constant $2.685\ldots$, as Khinchin's ergodic theorem predicts for almost every $x$; for a Liouville number it is infinite. The measure-typical number has moderate quotients, the Baire-typical number has wild ones.

### P5.5

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from math import gcd, sqrt

pts = [(p / q, 1 / q) for q in range(1, 61) for p in range(0, q + 1) if gcd(p, q) == 1]
plt.figure(figsize=(7, 3)); plt.plot(*zip(*pts), ".", ms=2); plt.title("Thomae's function, q <= 60"); plt.show()

x0 = sqrt(2) - 1
for k in range(1, 7):
    delta = 10.0 ** -k
    q = 1
    while True:                          # smallest q with a fraction p/q within delta of x0
        p = round(x0 * q)
        if abs(p / q - x0) < delta: break
        q += 1
    print(f"delta = 1e-{k}: max f = 1/{q}   (fraction {p}/{q})")

The maximum of $f$ on $]x_0-\delta,x_0+\delta[$ is $\frac1q$ for the smallest denominator $q$ of a fraction in that interval, and it tends to $0$: continuity at the irrational $x_0$. The fractions found, $\frac12,\frac5{12},\frac{12}{29},\frac{29}{70},\frac{169}{408},\frac{408}{985}$, are convergents of $\sqrt2-1=[0;2,2,2,\ldots]$, and $q$ grows like $(1+\sqrt2)^k$ while $\delta$ shrinks like $10^{-k}$; so near $\sqrt2-1$, $f$ is at most of order $\sqrt\delta$. Badly approximable numbers (bounded partial quotients) are the points where Thomae's function is the most continuous.

### P5.6

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(5)
digits = []
N = 2
while len(digits) < 2 * 10 ** 6:
    digits.extend(rng.integers(0, 2, 5 * N))          # an arbitrary open set: a few free digits
    n = len(digits)
    ones_needed = int(n * (N - 1)) + 1               # enough ones to push the frequency above 1 - 1/N
    digits.extend([1] * ones_needed)
    N += 1
digits = np.array(digits[:2 * 10 ** 6])
nu_typ = np.cumsum(digits) / np.arange(1, len(digits) + 1)
nu_rand = np.cumsum(rng.integers(0, 2, len(digits))) / np.arange(1, len(digits) + 1)
n = np.arange(1, len(digits) + 1)
plt.semilogx(n, nu_typ, label="Baire-typical"); plt.semilogx(n, nu_rand, label="random (normal)")
plt.ylim(0, 1.02); plt.legend(loc="lower right"); plt.xlabel("n"); plt.ylabel("proportion of ones"); plt.show()
print("min and max of nu_n over the second million digits:", round(float(nu_typ[10 ** 6:].min()), 6), round(float(nu_typ[10 ** 6:].max()), 6))

The construction is the proof of Baire's theorem made concrete: at stage $N$ the prefix already chosen is extended by a few free digits (which could be chosen to enter any prescribed open set) and then by enough ones to enter $V_N$. The proportion of ones climbs to $1-\frac1N$ at the end of each run, which is why $\limsup\nu_n=1$, while the random sequence settles at $\frac12$ as the law of large numbers says. With free blocks of zeros instead of random digits, the same procedure would also give $\liminf\nu_n=0$: a typical sequence in Baire's sense has no frequency at all. Here, because every run of ones is longer than everything before it, $\nu_n$ in fact tends to $1$ along the whole sequence; the comeagre set of Proposition 5.14 contains such points too.